In [ ]:
import os
import pickle
import time  # For debugging query time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location

testDates = [
    "2024-08-08",
    "2024-08-19",
    "2024-08-21",
    "2024-08-29",
    "2024-09-05",
    "2024-09-06",
    "2024-09-08",
    "2024-09-11",
    "2024-09-15",
    "2024-10-13",
    "2024-10-20",
    "2024-10-29",
    "2024-10-31",
    "2024-11-07",
    "2024-11-12",
    "2024-11-15",
    "2024-11-17",
    "2024-11-18",
    "2024-11-19",
    "2024-11-27",
    "2024-11-28",
    "2024-11-29",
    "2024-11-30",
    "2024-12-01",
    "2024-12-02",
    "2024-12-03",
    "2024-12-04",
    "2024-12-12",
    "2024-12-13",
    "2024-12-14",
    "2024-12-15",
    "2024-12-16",
    "2024-12-17",
    "2024-12-18",
    "2024-12-19",
    "2024-12-20",
    "2024-12-21",
    "2024-12-22",
    "2024-12-23",
    "2024-12-24",
    "2024-12-25",
    "2024-12-26",
    "2024-12-29",
    "2024-12-30",
    "2025-01-01",
    "2025-01-02",
    "2025-01-03",
    "2025-01-04",
    "2025-01-05",
    "2025-01-06",
    "2025-01-07",
    "2025-01-08",
    "2025-01-09",
    "2025-01-10",
    "2025-01-11",
    "2025-01-12",
    "2025-01-13",
    "2025-01-14",
    "2025-01-15",
    "2025-01-16",
    "2025-01-17",
    "2025-01-18",
    "2025-01-19",
    "2025-01-20",
    "2025-01-21",
    "2025-01-22",
    "2025-01-23",
    "2025-01-24",
    "2025-01-25",
    "2025-01-26",
    "2025-01-27",
    "2025-01-28",
    "2025-01-29",
    "2025-01-30",
    "2025-01-31",
    "2025-02-01",
    "2025-02-02",
    "2025-02-03",
    "2025-02-04",
    "2025-02-05",
    "2025-02-06",
    "2025-02-07",
    "2025-02-08",
    "2025-02-09",
    "2025-02-10",
    "2025-02-11",
    "2025-02-12",
    "2025-02-14",
    "2025-02-15",
    "2025-02-16",
    "2025-02-17",
    "2025-02-18",
    "2025-02-19",
]

# Create database connection
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}",
    connect_args={"connect_timeout": 3600},
    poolclass=QueuePool,
    pool_size=5,
    max_overflow=10,
)

# SQL query to retrieve data for a specific test date
query_template = """
SELECT 
    hr.test_date,
    ec.ech_public_name,
    COUNT(DISTINCT hr.id) AS record_count
FROM 
    public."HTTPSRecords" hr
JOIN 
    public."HTTPSRecordsECHConfigs" hrec ON hr.id = hrec.https_record_id
JOIN 
    public."ECHConfigs" ec ON hrec.ech_config_id = ec.id
WHERE 
    hr.ech_config != ''
    AND hr.test_date = %s  -- Use = for single test date
GROUP BY 
    hr.test_date, ec.ech_public_name
ORDER BY 
    hr.test_date, record_count DESC;
"""


def fetch_data(test_date):  # Accept a single test date
    """Fetches data for a single test date."""
    try:
        start_time = time.time()
        print(f"[START] Querying for test date: {test_date}")

        with engine.connect() as conn:
            # Pass the test date as a tuple
            result = pd.read_sql_query(query_template, conn, params=(test_date,))

        elapsed_time = time.time() - start_time
        print(f"[FINISH] Query complete for test date: {test_date} (Time taken: {elapsed_time:.2f} seconds)")
        return result
    except Exception as e:
        print(f"[ERROR] Fetching data for test date {test_date}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame on error


def save_data(test_date, group_df, output_dir):
    """Saves data for a specific test date to a pickle file."""
    try:
        print(f"[START] Saving data for test date {test_date}")
        os.makedirs(output_dir, exist_ok=True)
        filename = os.path.join(
            output_dir,
            f"{datetime.now().strftime('%Y-%m-%d_%HH-%MM-%SS')}_ech_records_{test_date}.pickle",
        )
        with open(filename, "wb") as f:
            pickle.dump(group_df, f)
        print(f"[FINISH] Saved data for test date {test_date} to {filename}")
    except Exception as e:
        print(f"[ERROR] Saving data for test date {test_date}: {e}")


def fetch_and_save_all_data(test_dates, output_dir="./pickle", workers=4):
    """Fetches and saves data for all test dates using multiple workers."""

    with ThreadPoolExecutor(max_workers=workers) as executor:
        # Query data for each test date in parallel
        results = list(executor.map(fetch_data, test_dates))  # No chunking needed

    # Combine all results into a single DataFrame
    all_data = pd.concat(results, ignore_index=True)

    # Save data for each test date
    for test_date, group_df in all_data.groupby("test_date"):
        save_data(test_date, group_df, output_dir)


if __name__ == "__main__":
    print("[INFO] Starting fetch and save process")
    fetch_and_save_all_data(testDates, workers=4)
    print("[INFO] Process complete")